# MLflow: make a model result traceable and reusable

Yesterday a digit classifier scored well, but today nobody can identify its preprocessing, training split or saved weights. We will build an experiment record that answers those questions: compare three configurations on validation data, select a candidate, evaluate it once on test data, reload it and give it a versioned local registry entry.

The project uses scikit-learn's bundled 8×8 digits. Everything is local: a SQLite tracking database, local artifacts and the Python API. No tracking server, account or dataset download is needed after setup. All explanations and examples are in this notebook.

The analytical habit is to distinguish a number from the evidence needed to reproduce and interpret it. Logging does not by itself prevent leakage or make an evaluation trustworthy.

## 1. Install the tracking client without changing other notebooks

This repository uses `mlflow-skinny==3.1.4`, imported as `mlflow`, plus SQLAlchemy, Alembic, scikit-learn, pandas, numpy, matplotlib and cloudpickle. The lightweight distribution supports the APIs demonstrated here without installing the full UI/server dependencies. It avoids the full MLflow 3.1.x package's older PyArrow constraint in this Python 3.9 environment.

For an independent environment, install `mlflow-skinny==3.1.4 sqlalchemy alembic scikit-learn pandas numpy matplotlib`. Use the repository pins when exact reproduction matters. The optional UI discussion later explains the full-package distinction.

Rerunning adds new runs and registry entries; it does not delete the tracking history. Summary CSV/JSON files under `outputs/mlflow/` are replaced.

In [ ]:
from pathlib import Path
import hashlib, json, uuid, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, ConfusionMatrixDisplay
from IPython.display import display

OUT = Path('outputs/mlflow').resolve(); OUT.mkdir(parents=True, exist_ok=True)
print({'python':platform.python_version(), 'mlflow':mlflow.__version__, 'sklearn':sklearn.__version__})
assert mlflow.active_run() is None, 'Finish your active run before starting this self-contained project.'

## 2. Separate experiment metadata from model files

A backend store holds run metadata such as parameters, metrics and tags. An artifact store holds files such as plots, model weights and reports. We use SQLite for metadata and an explicit local artifact directory, so the storage boundary is visible.

An experiment groups related runs. A run captures one execution, not one timeless model. Absolute local artifact paths are convenient here but are not portable by simply copying the SQLite database to another machine: the artifacts must move and remain addressable too.

In [ ]:
tracking_uri = 'sqlite:///' + str(OUT / 'tracking.sqlite3')
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()
experiment_name = 'digits-local-teaching'
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = client.create_experiment(experiment_name, artifact_location=(OUT / 'artifacts').as_uri())
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
batch_id = uuid.uuid4().hex[:12]
print({'tracking_uri':mlflow.get_tracking_uri(), 'experiment_id':experiment_id, 'comparison_batch':batch_id})

## 3. Record exactly which data supports the score

One row is a 64-pixel image; labels are digits 0–9. Split into approximately 60% training, 20% validation and 20% testing. Fit each candidate's scaler only on training data. Compare candidates on the same validation rows.

We save split indices and a SHA-256 digest of the source arrays. A digest identifies bytes; it does not grant access to missing data or prove that the labels are correct. The bundled dataset remains the source. MLflow dataset logging records lineage metadata; it is not a replacement for a governed data snapshot.

In [ ]:
SEED = 42
digits = load_digits()
X = pd.DataFrame(digits.data, columns=[f'pixel_{i:02d}' for i in range(64)])
y = digits.target
train_idx, remainder = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED)
valid_idx, test_idx = train_test_split(remainder, test_size=0.5, stratify=y[remainder], random_state=SEED)
assert set(train_idx).isdisjoint(valid_idx) and set(train_idx).isdisjoint(test_idx) and set(valid_idx).isdisjoint(test_idx)
fingerprint = hashlib.sha256(np.ascontiguousarray(digits.data).tobytes() + np.ascontiguousarray(y).tobytes()).hexdigest()
np.savez(OUT / 'split_indices.npz', train=train_idx, validation=valid_idx, test=test_idx)
training_frame = X.iloc[train_idx].copy(); training_frame['target'] = y[train_idx]
training_dataset = mlflow.data.from_pandas(training_frame, name='digits-training', targets='target')
display(pd.DataFrame({'split':['train','validation','test'], 'rows':[len(train_idx),len(valid_idx),len(test_idx)]}))

## 4. Decide what belongs in parameters, metrics, tags and artifacts

| Information | Record it as | Reason |
|---|---|---|
| Regularization C, seed, solver | Parameter | Describes a run's configuration |
| Validation log loss or F1 | Metric | Numeric result with a split-specific meaning |
| Dataset hash, purpose, batch ID | Tag | Searchable context |
| Confusion matrix, split IDs | Artifact | Structured or visual evidence |
| Input schema and example | Model signature / input example | Describes the inference interface |

Parameters should remain fixed within a run; create a new run for a different configuration. Metrics can have a step history. Name training, validation and test metrics explicitly so they cannot be mistaken for interchangeable scores.

Manual logging here makes each decision visible. Autologging can reduce repetitive code, but still inspect what it captures, especially when preprocessing and model selection occur in custom loops.

## 5. Establish a reference that uses no image structure

A prior-frequency classifier provides a weak but informative floor for probability quality. We choose validation log loss as the selection metric before fitting any candidate. Lower is better; macro F1 and accuracy remain useful secondary descriptions.

Predict whether a constant classifier can rank the digit classes differently for two different images. Its inability to use pixels is precisely why it is a useful reference, not a serious recognition system.

In [ ]:
prior = DummyClassifier(strategy='prior').fit(X.iloc[train_idx], y[train_idx])
prior_validation_loss = log_loss(y[valid_idx], prior.predict_proba(X.iloc[valid_idx]), labels=np.arange(10))
print('Prior baseline validation log loss:', prior_validation_loss)

## 6. Group one comparison under a parent run

The parent describes this comparison batch; child runs contain individual candidate settings. We compare C values 0.1, 1 and 10 using identical training and validation rows. A larger C means weaker regularization for logistic regression, not automatically a better model.

Log the complete preprocessing pipeline with a signature and example input. Specifying the model environment avoids guessing dependencies, but the environment files do not make arbitrary platforms identical. All model artifacts here originate locally; only load trusted Python-serialized models.

In [ ]:
model_infos = {}; fitted = {}; run_by_c = {}
with mlflow.start_run(run_name='compare-digit-classifiers', tags={'batch_id':batch_id, 'role':'comparison'}) as parent:
    parent_id = parent.info.run_id
    mlflow.log_params({'split_seed':SEED, 'training_rows':len(train_idx), 'validation_rows':len(valid_idx)})
    mlflow.set_tags({'dataset_sha256':fingerprint, 'selection_metric':'validation_log_loss', 'source':'MLflow.ipynb'})
    mlflow.log_metric('baseline_validation_log_loss', prior_validation_loss)
    mlflow.log_artifact(str(OUT / 'split_indices.npz'), artifact_path='data_contract')
    mlflow.log_input(training_dataset, context='training')
    for C in [0.1, 1.0, 10.0]:
        with mlflow.start_run(run_name=f'logistic-C-{C}', nested=True, tags={'batch_id':batch_id, 'role':'candidate'}) as run:
            candidate = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1200, random_state=SEED))
            candidate.fit(X.iloc[train_idx], y[train_idx])
            predicted = candidate.predict(X.iloc[valid_idx])
            probability = candidate.predict_proba(X.iloc[valid_idx])
            mlflow.log_params({'C':C, 'seed':SEED, 'max_iter':1200, 'preprocessing':'StandardScaler', 'solver':'lbfgs'})
            mlflow.set_tag('dataset_sha256',fingerprint)
            mlflow.log_metrics({'validation_log_loss':log_loss(y[valid_idx],probability,labels=np.arange(10)),
                                'validation_accuracy':accuracy_score(y[valid_idx],predicted),
                                'validation_macro_f1':f1_score(y[valid_idx],predicted,average='macro')})
            signature = infer_signature(X.iloc[train_idx[:5]], candidate.predict(X.iloc[train_idx[:5]]))
            info = mlflow.sklearn.log_model(candidate, name='pipeline', signature=signature,
                                           input_example=X.iloc[train_idx[:2]],
                                           pip_requirements=[f'scikit-learn=={sklearn.__version__}',
                                                             f'numpy=={np.__version__}', f'pandas=={pd.__version__}',
                                                             'cloudpickle==3.1.2'])
            model_infos[run.info.run_id] = info
            fitted[run.info.run_id] = candidate
            run_by_c[C] = run.info.run_id
assert mlflow.active_run() is None
print('Parent run:', parent_id)

## 7. Query the record instead of relying on notebook memory

Search only candidates from this comparison batch. Without that filter, a rerun could accidentally select a model from an older split or earlier experiment. Sort by the predeclared validation metric and use run ID only as a deterministic tie-breaker.

Check that there are exactly three completed candidates and that each has a loss. Missing metrics should not silently participate in model selection.

In [ ]:
runs = mlflow.search_runs(experiment_ids=[experiment_id],
    filter_string=f"tags.batch_id = '{batch_id}' and tags.role = 'candidate' and attributes.status = 'FINISHED'",
    order_by=['metrics.validation_log_loss ASC'])
assert len(runs) == 3 and runs['metrics.validation_log_loss'].notna().all()
runs = runs.sort_values(['metrics.validation_log_loss','run_id'])
display(runs[['run_id','params.C','metrics.validation_log_loss','metrics.validation_accuracy','metrics.validation_macro_f1']])
best_run_id = runs.iloc[0].run_id
best_uri = model_infos[best_run_id].model_uri
print('Selected model URI:',best_uri)

## 8. Evaluate the selected artifact once on the held-out images

Reload the selected logged pipeline before test evaluation. This verifies the artifact that a later consumer will use, rather than only the in-memory object that trained successfully. Compare its predictions with the original and report the prior baseline on the same test rows.

Log test metrics in a separate evaluation run linked to the selected training run. Testing every new idea repeatedly would turn this test set into another validation set; keeping a log does not prevent that mistake.

In [ ]:
loaded = mlflow.sklearn.load_model(best_uri)
assert np.array_equal(loaded.predict(X.iloc[valid_idx]), fitted[best_run_id].predict(X.iloc[valid_idx]))
predicted = loaded.predict(X.iloc[test_idx]); probabilities = loaded.predict_proba(X.iloc[test_idx])
test_metrics = {'test_accuracy':float(accuracy_score(y[test_idx],predicted)),
                'test_macro_f1':float(f1_score(y[test_idx],predicted,average='macro')),
                'test_log_loss':float(log_loss(y[test_idx],probabilities,labels=np.arange(10))),
                'baseline_test_log_loss':float(log_loss(y[test_idx],prior.predict_proba(X.iloc[test_idx]),labels=np.arange(10)))}
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y[test_idx], predicted, ax=ax)
plt.tight_layout(); plt.show()
with mlflow.start_run(run_name='held-out-evaluation', tags={'batch_id':batch_id, 'role':'evaluation', 'selected_run_id':best_run_id}) as evaluation:
    evaluation_id = evaluation.info.run_id
    mlflow.log_metrics(test_metrics)
    mlflow.log_figure(fig, 'confusion_matrix.png')
    mlflow.log_dict({'source_model_uri':best_uri, 'source_run_id':best_run_id,
                     'test_indices':test_idx.tolist(), 'dataset_sha256':fingerprint}, 'evaluation_contract.json')
display(pd.Series(test_metrics))

## 9. Use the model signature as part of a contract

The logged signature names 64 floating-point pixel columns and a predicted-label output. MLflow's Python-function flavor offers a common prediction interface. It checks structural compatibility, but it does not know that pixel values must be 0–16 unless your application validates that semantic constraint.

Here the output is a class label because the default scikit-learn pyfunc method is `predict`. Do not assume it returns a probability vector. Document which method and output type consumers receive.

In [ ]:
portable = mlflow.pyfunc.load_model(best_uri)
example_input = X.iloc[test_idx[:5]]
assert list(example_input.columns) == list(X.columns)
assert ((example_input.to_numpy() >= 0) & (example_input.to_numpy() <= 16)).all()
portable_predictions = portable.predict(example_input)
assert np.array_equal(portable_predictions, loaded.predict(example_input))
print('Signature:', portable.metadata.signature)
print('Predictions:', portable_predictions)

## 10. Create a versioned local reference without deploying anything

A registered model groups versions under a name. An alias points to a chosen version and can change later. We create a unique teaching model name and assign a `candidate` alias; this does not start a server or promote the model into any production service.

Record the exact numeric version for an immutable handoff. An alias is useful for controlled updates, but reproducing yesterday's behavior requires knowing which version it referenced yesterday.

In [ ]:
registry_name = 'digits-teaching-' + batch_id
version = mlflow.register_model(model_uri=best_uri, name=registry_name)
client.set_registered_model_alias(registry_name, 'candidate', version.version)
resolved = client.get_model_version_by_alias(registry_name, 'candidate')
assert resolved.version == version.version
registry_uri = f'models:/{registry_name}/{version.version}'
registered = mlflow.pyfunc.load_model(registry_uri)
assert np.array_equal(registered.predict(example_input), portable_predictions)
print({'registry_name':registry_name, 'version':version.version, 'alias':'candidate'})

## 11. Find the evidence behind an artifact

Retrieve a candidate run by ID and inspect parameters, metrics, and parent linkage. Metrics retrieved directly from a run show latest values; for an iterative metric use `get_metric_history` to inspect steps. Do not fabricate a training curve by repeatedly logging the same final number.

Export a compact handoff containing the tracking URI, run IDs, model URI, registry version and dataset fingerprint. Local paths remain machine-specific. To share reproducibly, use a supported shared tracking/artifact setup or move the complete required artifacts with an explicit migration plan.

In [ ]:
record = client.get_run(best_run_id)
assert record.data.tags['mlflow.parentRunId'] == parent_id
history = client.get_metric_history(best_run_id, 'validation_log_loss')
print('Stored parameters:',record.data.params)
print('Metric steps:',[(m.step,m.value) for m in history])
handoff = {'tracking_uri':tracking_uri, 'experiment_id':experiment_id, 'parent_run_id':parent_id,
           'selected_run_id':best_run_id, 'evaluation_run_id':evaluation_id, 'model_uri':best_uri,
           'registry_uri':registry_uri, 'dataset_sha256':fingerprint, 'test_metrics':test_metrics}
(OUT / 'handoff.json').write_text(json.dumps(handoff,indent=2))
runs.to_csv(OUT / 'candidate_comparison.csv',index=False)
print('Handoff saved:',OUT / 'handoff.json')

## 12. Know what a server and UI add

The notebook uses the lightweight client and local stores. A full MLflow installation adds UI/server dependencies. For a separate compatible full environment, a local UI can be started with a command such as:

```sh
mlflow ui --backend-store-uri sqlite:///outputs/mlflow/tracking.sqlite3 --host 127.0.0.1 --port 5000
```

Run from this repository directory and use an MLflow version compatible with the created database. The full 3.1.x package constrains PyArrow below this repository's pin; do not install it over this environment casually. The notebook's Python searches provide the required inspection without a UI.

For a team, separate client machines, tracking service, metadata database and artifact storage. Authentication, access control, retention and backups become operational requirements. Run history is not the same as source control, dataset versioning, model quality monitoring or deployment approval.

Optuna chooses configurations; MLflow records experiments. They can be combined by logging each completed/pruned trial with its parameters and state. Keep objective values, metric names and parent/child IDs aligned so that a partially trained trial is not mistaken for a selected model.

## 13. Explain what the evidence supports

1. Starting only with `handoff.json`, identify the selected C, dataset fingerprint, validation score and exact registry version. Which extra evidence would be needed to recreate the training environment on another machine?
2. Deliberately remove a required column from a copy of the inference input and inspect the resulting contract error. Separately explain why a value of 999 may need application validation even if its type matches.
3. Add a new candidate in a new comparison batch. Choose by validation and reserve fresh final evaluation data if model development continues.
4. Describe what an alias change means for a consumer that loads by alias versus one that loads by version.

| Observation | Investigate |
|---|---|
| Runs appear absent | Tracking URI, experiment ID and search filter |
| Selected artifact cannot be loaded | Artifact path availability and environment compatibility |
| Parameter cannot be changed | Whether a new configuration needs a new run |
| Another run becomes nested accidentally | Active-run lifetime and context managers |
| Excellent metrics cannot be reproduced | Data snapshot, split IDs, preprocessing and source version |

Official references: [tracking](https://mlflow.org/docs/3.1.3/ml/tracking/), [model registry](https://mlflow.org/docs/3.1.3/ml/model-registry/), [model signatures](https://mlflow.org/docs/3.1.3/ml/model/signatures/), and [Python API](https://mlflow.org/docs/3.1.3/api_reference/python_api/mlflow.html). These versioned references are close to the tested patch version.

## Second project: detect incomparable runs before selecting a model

A subscription-scoring export has repeated rows, malformed and missing usage, missing labels, a new category appearing late, and a tempting field populated only after the outcome. Build a time-ordered development/test workflow on 1,000 generated records with a mild late-period shift. This is a synthetic operational scenario, not a real churn dataset; it is designed to expose boundary mistakes while keeping training small.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a feature that predicts too well

`refund_after_outcome` is generated from the label. It would look highly useful but is unavailable when a churn prediction is made. Exclude it by an explicit feature allowlist before preprocessing or tuning. Also exclude row IDs, which index time but are not intended predictors here. Missing labels cannot be imputed as though they were ordinary features.

Fit imputation, scaling and categorical vocabulary only on the training period. An unseen late category should be handled deliberately rather than silently refitting the encoder on the test set.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(63);p2_n=1000
p2_usage=p2_rng.gamma(2,3,p2_n);p2_tenure=p2_rng.integers(1,36,p2_n)
p2_region=p2_rng.choice(['East','West','North'],p2_n).astype(object);p2_region[900:]='NewRegion'
p2_logit=1.8-.28*p2_usage-.035*p2_tenure+.5*(p2_region=='West')+.3*(np.arange(p2_n)>=800)
p2_target=p2_rng.binomial(1,1/(1+np.exp(-p2_logit)))
p2_raw=pd.DataFrame({'row_id':np.arange(p2_n),'usage':p2_usage.astype(object),'tenure':p2_tenure,
 'region':p2_region,'churn':p2_target.astype(float),'refund_after_outcome':p2_target+np.random.default_rng(64).normal(0,.01,p2_n)})
p2_raw.loc[10:19,'usage']='malformed';p2_raw.loc[20:39,'usage']=None;p2_raw.loc[3:7,'churn']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().copy();assert p2_clean.row_id.is_unique
p2_clean['usage']=pd.to_numeric(p2_clean.usage,errors='coerce')
p2_missing_label=int(p2_clean.churn.isna().sum());p2_clean=p2_clean.dropna(subset=['churn'])
p2_clean['churn']=p2_clean.churn.astype(int)
p2_train=p2_clean[p2_clean.row_id<600];p2_valid=p2_clean[(p2_clean.row_id>=600)&(p2_clean.row_id<800)];p2_test=p2_clean[p2_clean.row_id>=800]
p2_features=['usage','tenure','region'];assert 'refund_after_outcome' not in p2_features
assert len(p2_train)==595 and len(p2_valid)==len(p2_test)==200
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss,roc_auc_score,average_precision_score

def p2_preprocess():
    return ColumnTransformer([('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),['usage','tenure']),
      ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['region'])],sparse_threshold=0)
def p2_pipeline(estimator):return Pipeline([('prepare',p2_preprocess()),('model',estimator)])

import mlflow,mlflow.sklearn,json,hashlib,uuid
from mlflow import MlflowClient
from mlflow.models import infer_signature
p2_tracking='sqlite:///'+str((p2_out/'mlflow_comparison.sqlite3').resolve())
mlflow.set_tracking_uri(p2_tracking)
p2_client=MlflowClient();p2_experiment='project2-contracts'
p2_experiment_record=p2_client.get_experiment_by_name(p2_experiment)
p2_eid=(p2_experiment_record.experiment_id if p2_experiment_record else p2_client.create_experiment(p2_experiment,artifact_location=(p2_out/'mlflow_artifacts').resolve().as_uri()))
p2_batch=uuid.uuid4().hex[:8]
p2_fingerprint=hashlib.sha256(p2_train[['row_id','churn']].to_csv(index=False).encode()).hexdigest()
p2_fit=p2_pipeline(LogisticRegression(max_iter=1000)).fit(p2_train[p2_features],p2_train.churn)
p2_real_loss=log_loss(p2_valid.churn,p2_fit.predict_proba(p2_valid[p2_features])[:,1])
# The second record intentionally represents an incompatible reporting contract; its attractive number is not a model result.
for p2_contract,p2_metric,p2_demo in [('temporal-v1',p2_real_loss,False),('random-split-v0',.01,True)]:
    with mlflow.start_run(experiment_id=p2_eid,tags={'batch':p2_batch,'evaluation_contract':p2_contract,'fixture_metric':str(p2_demo)}) as p2_run:
        mlflow.log_metric('validation_log_loss',float(p2_metric))
        mlflow.set_tag('training_split_sha256',p2_fingerprint if not p2_demo else 'different-split')
        mlflow.log_params({'feature_count':3,'model':'logistic','seed':63})
        if not p2_demo:
            p2_good_run=p2_run.info.run_id
            p2_info=mlflow.sklearn.log_model(p2_fit,name='pipeline',signature=infer_signature(p2_train[p2_features],p2_fit.predict(p2_train[p2_features])),
                input_example=p2_train[p2_features].iloc[:2],pip_requirements=['scikit-learn==1.6.1','pandas==2.3.3','numpy==1.26.4','cloudpickle==3.1.2'])

### Work through the comparison contract

Sorting all runs by a field called validation loss chooses the fabricated 0.01, even though it represents another split. Filter by the explicit evaluation contract, batch and dataset fingerprint before ranking. A tracking system stores claims; it does not know whether two numbers measure the same thing.

For artifact cost, save a tiny prediction sample and model contract rather than logging the entire training table every run. A digest identifies data but cannot recover it; preserve the approved data snapshot separately when reproducibility requires it.

In [ ]:
p2_all=mlflow.search_runs([p2_eid],filter_string=f"tags.batch = '{p2_batch}'",order_by=['metrics.validation_log_loss ASC'])
assert p2_all.iloc[0]['tags.fixture_metric']=='True'
p2_comparable=p2_all[(p2_all['tags.evaluation_contract']=='temporal-v1')&(p2_all['tags.training_split_sha256']==p2_fingerprint)]
assert len(p2_comparable)==1 and p2_comparable.iloc[0].run_id==p2_good_run
p2_loaded=mlflow.sklearn.load_model(p2_info.model_uri)
p2_test_probability=p2_loaded.predict_proba(p2_test[p2_features])[:,1]
assert np.allclose(p2_test_probability,p2_fit.predict_proba(p2_test[p2_features])[:,1])
p2_contract={'selected_run':p2_good_run,'feature_order':p2_features,'excluded_future_field':'refund_after_outcome',
 'training_split_sha256':p2_fingerprint,'test_log_loss':float(log_loss(p2_test.churn,p2_test_probability))}
with mlflow.start_run(run_id=p2_good_run):
    mlflow.log_dict(p2_contract,'comparison_contract.json')
    mlflow.log_metric('test_log_loss',p2_contract['test_log_loss'])
p2_all[['run_id','metrics.validation_log_loss','tags.evaluation_contract','tags.fixture_metric']].to_csv(p2_out/'mlflow_run_audit.csv',index=False)
display(p2_all[['metrics.validation_log_loss','tags.evaluation_contract','tags.fixture_metric']]);print(p2_contract)
print({'full_training_csv_bytes':len(p2_train.to_csv(index=False).encode()),'contract_json_bytes':len(json.dumps(p2_contract).encode())})

### Explain the debugging outcome

The lowest stored number was deliberately ineligible. The filtered record identifies a reloadable model and a particular temporal comparison. Tags are useful checks, but callers can mistag runs; high-assurance pipelines generate contracts from actual split manifests and verify them.

**Try:** omit the fingerprint tag from a new record. **Worked reasoning:** it should be excluded from the comparable set or flagged for review, not silently treated as equivalent. Reducing logged bytes by storing only metadata is useful only if the underlying data snapshot remains available through a documented, permitted path.